# Predicting Developer Salaries

Stack Overflow Developer Survey 2025 subsample, 5,000 responses. Target: `annual_salary_usd`.

**How to use this notebook:** cells marked `TODO(you)` are decisions, not typing. Each one says what
you're deciding and why it matters. The mechanical cells are filled in so you spend your time on
judgment instead of boilerplate.

Restart & Run All before every commit. A notebook whose outputs don't match its code is worse than
no notebook.

## 0. Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

RANDOM_SEED = 0
N_FOLDS = 5

df_raw = pd.read_csv("data/survey.csv")
assert df_raw.shape == (5000, 16), f"unexpected shape: {df_raw.shape}"
print(f"{df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
df_raw.head(3)

5,000 rows x 16 columns


,ResponseId,Age,EdLevel,Employment,WorkExp,YearsCode,DevType,OrgSize,ICorPM,RemoteWork,Industry,Country,Currency,LanguageHaveWorkedWith,DatabaseHaveWorkedWith,annual_salary_usd
0,1,25-34 years old,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",Employed,8.0,14.0,"Developer, mobile",20 to 99 employees,People manager,Remote,Fintech,Ukraine,EUR European Euro,Bash/Shell (all shells);Dart;SQL,Cloud Firestore;PostgreSQL,61256.0
1,10004,55-64 years old,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",Employed,40.0,46.0,"Developer, full-stack","10,000 or more employees",Individual contributor,"Your choice (very flexible, you can come in wh...",Computer Systems Design and Services,Germany,EUR European Euro,C#;Java;SQL,H2;MariaDB;Neo4J;Oracle,99773.0
2,10016,25-34 years old,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",Employed,2.0,5.0,"Developer, full-stack","10,000 or more employees",Individual contributor,Remote,Healthcare,United States of America,USD United States dollar,Bash/Shell (all shells);C#;HTML/CSS;JavaScript...,Microsoft SQL Server;MongoDB,85000.0


---
# 1. Column-by-column audit

Look at everything before changing anything. Cleaning destroys evidence: once you drop the $1
salaries you can no longer investigate whether they were monthly figures.

In [2]:
def audit(df):
    """Print dtype, completeness, cardinality and top values for every column."""
    for col in df.columns:
        s = df[col]
        pct = s.notna().mean()
        print(f"\n{'='*78}\n{col}   dtype={s.dtype}   non-null={s.notna().sum():,}/{len(s):,} "
              f"({pct:.1%})   distinct={s.nunique(dropna=True):,}")
        if s.dtype.kind in "if":
            print(s.describe().to_string())
        else:
            print(s.value_counts(dropna=False).head(8).to_string())

audit(df_raw)


ResponseId   dtype=int64   non-null=5,000/5,000 (100.0%)   distinct=5,000
count     5000.000000
mean     22275.489200
std      14270.069609
min          1.000000
25%      10242.250000
50%      20486.000000
75%      35120.750000
max      49181.000000

Age   dtype=str   non-null=5,000/5,000 (100.0%)   distinct=7
Age
25-34 years old      1892
35-44 years old      1578
45-54 years old       664
18-24 years old       545
55-64 years old       270
65 years or older      45
Prefer not to say       6

EdLevel   dtype=str   non-null=4,995/5,000 (99.9%)   distinct=8
EdLevel
Bachelor’s degree (B.A., B.S., B.Eng., etc.)                                          2283
Master’s degree (M.A., M.S., M.Eng., MBA, etc.)                                       1464
Some college/university study without earning a degree                                 578
Professional degree (JD, MD, Ph.D, Ed.D, etc.)                                         242
Secondary school (e.g. American high school, German Realschule o

## Column verdicts

`KEEP` (use as-is) / `CLEAN` (fix something first) / `ENGINEER` (needs encoding) / `DROP`.
Counts come from `audit(df_raw)` above, so they describe all 5,000 rows before any cleaning.

| Column | Verdict | Why |
|---|---|---|
| `ResponseId` | DROP | 5,000 distinct values in 5,000 rows, so it is an identifier, not a feature -- a tree handed it would memorise rows. `clean()` keeps it in `df` only so section 9 can label predictions, and `build_features()` never selects it. It is also why the dedup excludes it: it hides the one otherwise-identical pair. |
| `Age` | ENGINEER | 7 levels -- 6 ordered bands plus "Prefer not to say" (5 rows). Ordinal 0-5, with the non-answer sent to NaN rather than placed somewhere on the scale. |
| `EdLevel` | ENGINEER | 8 levels -- 7 rungs of a ladder plus "Other (please specify):" (48 rows), and 5 blanks. Ordinal 0-6, "Other" and blanks to NaN. Two level names contain a curly apostrophe (U+2019), so the order list must match the CSV exactly; a one-character typo would silently turn the column into missing data, which is what the assert in section 4 guards. |
| `Employment` | ENGINEER | 6 unordered levels, no blanks. Category dtype. Also the column that explains most of the missingness elsewhere -- `OrgSize`, `ICorPM` and `RemoteWork` are blank largely for people who are not conventionally employed, so the model needs this one to read those NaNs correctly. |
| `WorkExp` | KEEP | Float years, 52 blanks, range 1-100. No imputation: HGB splits on NaN natively. The 7 rows above 50 years are implausible but too few to move anything, and 199 rows report more work experience than years coding -- possible if that experience was not all coding, so left alone rather than "corrected" on a guess. |
| `YearsCode` | KEEP | Same treatment. Float, 21 blanks, range 1-100, 9 rows above 50. |
| `DevType` | ENGINEER | 32 unordered levels, no blanks. Category dtype with every level kept -- the 9 levels holding under 20 rows cover just 74 rows between them, a small enough share to leave alone. |
| `OrgSize` | ENGINEER | 9 levels, but only 8 are positions on a size scale. "I don't know" (47 rows, curly apostrophe again) is a non-answer, not a size. Ordinal over the 8, with that level and the 530 blanks going to NaN. |
| `ICorPM` | ENGINEER | Only 2 real values against 602 blanks. Category dtype, and the NaN is informative rather than missing: it marks people the question does not apply to, which `Employment` already accounts for. |
| `RemoteWork` | ENGINEER | 5 unordered levels, 553 blanks. Category dtype -- "Hybrid" and "Your choice" have no defensible rank between them, so an ordinal would assert an order that is not there. |
| `Industry` | ENGINEER | 15 unordered levels, 144 blanks. Category dtype, no natural order. |
| `Country` | ENGINEER | 130 levels, no blanks. Category dtype, ungrouped -- well under HGB's 255-category ceiling. The strongest single predictor by a wide margin: the country-median baseline in section 5 reaches most of the way to the full model on this column alone. |
| `Currency` | CLEAN | 97 values, and 1,133 rows separate the 3-letter code from the currency name with a tab instead of a space, so the raw string is an unreliable key. Split on whitespace, keep `currency_code`, drop the original. |
| `LanguageHaveWorkedWith` | ENGINEER | Semicolon-joined lists: 2,940 distinct strings collapse to 42 tokens. Multi-hot, one column per language. The 349 blanks become all-zero rows. |
| `DatabaseHaveWorkedWith` | ENGINEER | Same shape -- 1,672 strings collapse to 30 tokens. Caveat: the 1,052 blanks also become all-zero, so "did not answer" and "uses no database" are indistinguishable to the model. |
| `annual_salary_usd` | TARGET | 84 blanks, range $1 to $6,890,299, and only 2,103 distinct values across 4,916 answers -- far too few for a continuous field, which is the thread that leads to the repeated EUR figures two cells down. Rows with no target are dropped, the rest trimmed to $10k-$400k, and the model fits `log1p` of it. |


---
# 2. The target

Everything downstream depends on understanding this column first.

In [3]:
y_raw = df_raw.annual_salary_usd

print("missing:", y_raw.isna().sum(), "rows\n")
print(y_raw.describe().to_string())
print("\nquantiles:")
print(y_raw.quantile([.001, .01, .05, .25, .5, .75, .95, .99, .999]).to_string())
print(f"\nbelow $1,000: {(y_raw < 1_000).sum()}    below $5,000: {(y_raw < 5_000).sum()}"
      f"    above $1M: {(y_raw > 1_000_000).sum()}")

missing: 84 rows

count    4.916000e+03
mean     9.835268e+04
std      1.454344e+05
min      1.000000e+00
25%      4.176500e+04
50%      7.755900e+04
75%      1.239930e+05
max      6.890299e+06

quantiles:
0.001          3.0
0.010         85.6
0.050       3988.0
0.250      41765.0
0.500      77559.0
0.750     123993.0
0.950     232029.0
0.990     500000.0
0.999    1291668.7

below $1,000: 124    below $5,000: 282    above $1M: 8


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
y_raw.dropna().plot.hist(bins=100, ax=ax[0], title="Raw salary (unreadable)")
np.log1p(y_raw.dropna()).plot.hist(bins=100, ax=ax[1], title="log1p(salary)")
ax[0].set_xlabel("USD"); ax[1].set_xlabel("log1p(USD)")
plt.tight_layout()

print(f"raw skew   {y_raw.skew():>7.2f}")
print(f"log1p skew {np.log1p(y_raw).skew():>7.2f}")

### TODO(you): look at the tails yourself

Print the 15 largest and 15 smallest salaries alongside `Country` and `Currency`. Then decide,
in writing, whether each tail is real data or broken data — and how you can tell.

Hint: `CompFreq` (weekly / monthly / yearly) is **not** in this subsample. Think about what that
means for your ability to repair a suspiciously small number.

In [ ]:
# TODO(you): inspect both tails
# cols = ["Country", "Currency", "Employment", "annual_salary_usd"]
# df_raw.nlargest(15, "annual_salary_usd")[cols]
# df_raw.nsmallest(15, "annual_salary_usd")[cols]

In [ ]:
# A continuous salary field should have almost no exact duplicates. This one does.
print(y_raw.value_counts().head(8).to_string())

print("\ncurrency of the 3 most repeated values:")
top3 = y_raw.value_counts().head(3).index
print(df_raw[df_raw.annual_salary_usd.isin(top3)]
      .groupby(["annual_salary_usd", "Currency"]).size().to_string())

### TODO(you): explain the cell above

66 respondents report the *exact* same salary to the dollar, and they all use the same currency.
That does not happen by chance.

Work out what produced it. Two questions to get you there:

1. Divide the largest of those repeated values by the second largest. What kind of number is that ratio?
2. If you divided every EUR salary by ~1.16, what would the results look like?

Write what you conclude here. It shapes two decisions later, so get it straight now.

---
# 3. Cleaning

One function, every threshold a named constant, every line justified. This is the code most likely
to get read aloud in your interview.

### The trim decision, measured before you make it

Don't guess thresholds. Run the cell below: it holds the model fixed and changes only the filter,
so the difference you see is caused by the filter alone.

Then pick your thresholds *from this table* and defend them with it.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import KFold, cross_val_predict

_CATS = ["Age", "EdLevel", "Employment", "DevType", "OrgSize",
         "RemoteWork", "Industry", "Country"]

def _quick_X(d):
    """Minimal feature frame, only for the sensitivity test below."""
    X = d[_CATS].astype("category").copy()
    X["WorkExp"] = d.WorkExp.values
    X["YearsCode"] = d.YearsCode.values
    return X

def trim_sensitivity(df, rules):
    base = df.dropna(subset=["annual_salary_usd"])
    kf = KFold(N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
    rows = []
    for name, lo, hi in rules:
        d = base[base.annual_salary_usd.between(lo, hi)]
        y = np.log1p(d.annual_salary_usd.values)
        X = _quick_X(d)
        m = HistGradientBoostingRegressor(
            categorical_features=[str(t) == "category" for t in X.dtypes],
            random_state=RANDOM_SEED, max_iter=300, learning_rate=0.06)
        p = cross_val_predict(m, X, y, cv=kf)
        actual, pred = np.expm1(y), np.expm1(p)
        rows.append({
            "rule": name, "n": len(d),
            "MAE": np.abs(actual - pred).mean(),
            "MedAE": np.median(np.abs(actual - pred)),
            "R2_log": 1 - ((y - p) ** 2).sum() / ((y - y.mean()) ** 2).sum(),
        })
    return pd.DataFrame(rows).set_index("rule")

RULES = [
    ("keep everything",  0,      np.inf),
    ("drop < $1k",       1_000,  np.inf),
    ("drop < $5k",       5_000,  np.inf),
    ("$5k - $500k",      5_000,  500_000),
    ("$10k - $400k",     10_000, 400_000),
]
trim_sensitivity(df_raw, RULES).round({"MAE": 0, "MedAE": 0, "R2_log": 3})

### TODO(you): write `clean()`

Decisions to make, each with a `# why:` comment on the line:

1. **Rows with no target.** 84 of them. Can you train on them? Can you score them?
2. **Your trim thresholds.** Pick from the table above. Name them as constants.
3. **The duplicate.** One exact dup ignoring `ResponseId`.
4. **`Currency`.** 1,133 values use a tab instead of a space. Normalize to the 3-letter code.
5. **Columns to drop.** `ResponseId` is one. Is there anything else you can justify dropping?

Return a *new* frame. Don't mutate the input — you'll want `df_raw` intact for comparison.

In [ ]:
MIN_SALARY = 10_000    # why: the floor is what fixes R2_log -- 0.326 -> 0.502 -> 0.514 as it rises $0 -> $1k -> $5k at a fixed ceiling
MAX_SALARY = 400_000   # why: the ceiling is what fixes MAE -- at a fixed $5k floor, capping cut MAE $39,519 -> $30,982

def clean(df):
    df = df.copy()                                                          # why: caller keeps df_raw intact for the before/after comparison
    df = df[df.annual_salary_usd.notna()]                                   # why: 84 rows have no y -- unscorable, and imputing a target invents the answer
    df = df[df.annual_salary_usd.between(MIN_SALARY, MAX_SALARY)]           # why: .between() is False for NaN, so this would silently absorb the line above; separating them keeps the 84 attributable
    df = df.drop_duplicates(subset=df.columns.drop("ResponseId"))           # why: ResponseId is unique by construction, so it hides the one otherwise-identical pair (4119 / 15999)
    df["currency_code"] = df.Currency.str.split(r"\s+", regex=True).str[0]  # why: 1,133 of 5,000 rows separate code from name with a tab, so the raw string is an unreliable key
    # flagged: ResponseId stays because section 9 writes it to predictions.csv; ICorPM stays because
    # +0.0006 importance is a feature-selection call, not a data defect -- build_features() never selects it.
    return df.drop(columns=[
        "Currency",   # why: currency_code holds the same 97 levels behind a stable key, so the raw string is dead weight
    ])

df = clean(df_raw)
print(f"{len(df_raw):,} rows -> {len(df):,} rows  ({len(df_raw) - len(df):,} removed)")

In [ ]:
# Each assert states a guarantee clean() is supposed to make. If one fires the pipeline stops
# here, rather than training on a frame that no longer means what the code above says it means.
# _target, not y: from section 4 onward `y` means log1p dollars, and one name per meaning.

_target = df.annual_salary_usd

assert _target.notna().all(), f"{_target.isna().sum()} null targets survived the dropna"
assert _target.between(MIN_SALARY, MAX_SALARY).all(), \
    f"target escaped the trim: min={_target.min():,.0f} max={_target.max():,.0f}"
assert not df.drop(columns="ResponseId").duplicated().any(), \
    f"{df.drop(columns='ResponseId').duplicated().sum()} duplicate rows survived"
assert 4_300 <= len(df) <= 4_500, \
    f"row count {len(df):,} is outside the band this trim produces -- a filter changed meaning"
# ^ recompute the band if you change MIN_SALARY / MAX_SALARY. Its job is to catch a filter that
#   silently stopped filtering, which a spot-check of df.head() would never show.

_kept = df_raw.dropna(subset=["annual_salary_usd"])
_trim = _kept[_kept.annual_salary_usd.between(MIN_SALARY, MAX_SALARY)]
print(f"before : {len(df_raw):,}")
print(f"after  : {len(df):,}  ({len(df_raw) - len(df):,} removed, {1 - len(df) / len(df_raw):.1%})")
print(f"  null target     -{len(df_raw) - len(_kept):>4}")
print(f"  outside trim    -{len(_kept) - len(_trim):>4}")
print(f"  exact duplicate -{len(_trim) - len(df):>4}")

---
# 4. Features

Encode by column *kind*. Numeric needs nothing, ordered categories get integer codes in the real
order, unordered categories stay categorical, list-valued columns get multi-hot.

*(FUNDAMENTALS.md Part 2 if any of that is fuzzy.)*

In [ ]:
# Ordered categories. EdLevel is done as the pattern -- you do the other two.

ED_ORDER = [
    "Primary/elementary school",
    "Secondary school (e.g. American high school, German Realschule or Gymnasium, etc.)",
    "Some college/university study without earning a degree",
    "Associate degree (A.A., A.S., etc.)",
    "Bachelor’s degree (B.A., B.S., B.Eng., etc.)",
    "Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",
    "Professional degree (JD, MD, Ph.D, Ed.D, etc.)",
]
# note the curly apostrophe U+2019 -- these strings must match the CSV exactly.
# verify: set(ED_ORDER) - set(df_raw.EdLevel.dropna())   # should print set()

# OrgSize has 9 distinct values, but only 8 of them are positions on a size scale.
ORG_ORDER = [
    "Just me - I am a freelancer, sole proprietor, etc.",   # why: a company of one is the smallest size there is, so it anchors the scale rather than sitting off it
    "Less than 20 employees",
    "20 to 99 employees",
    "100 to 499 employees",
    "500 to 999 employees",
    "1,000 to 4,999 employees",
    "5,000 to 9,999 employees",
    "10,000 or more employees",
]
# why: "I don't know" (39 rows, spelt with U+2019 in the CSV) is deliberately absent -- it is a
# non-answer, not a size, and every position on the scale would assert something false about it.
# ordinal() sends it to NaN instead, which HGB splits on natively.

AGE_ORDER = [
    "18-24 years old", "25-34 years old", "35-44 years old",
    "45-54 years old", "55-64 years old", "65 years or older",
]
# why: "Prefer not to say" (5 rows) absent for the same reason -- no age band it belongs beside.

def ordinal(series, order):
    """Map an ordered category to 0..n-1. Values outside `order` become NaN."""
    return series.map({v: i for i, v in enumerate(order)}).astype("float")

In [ ]:
def multi_hot(series, prefix):
    """Split a ';'-joined column into one 0/1 column per token."""
    return series.fillna("").str.get_dummies(";").add_prefix(prefix)

# sanity check: raw-string cardinality vs token count
_lang = df_raw.LanguageHaveWorkedWith.fillna("")
print(f"raw string as a category : {_lang.nunique():>5} values")
print(f"multi-hot tokens         : {multi_hot(_lang, 'lang_').shape[1]:>5} columns")

### TODO(you): build `X` and `y`

Assemble the feature frame. Decisions:

- Which unordered categoricals to keep as `category` dtype (the model handles these natively).
- Whether to collapse `DevType`'s long tail — 11 of its 32 levels have fewer than 20 rows.
- Whether missing categoricals become a literal `"Missing"` level or stay `NaN`.
  Run `pd.crosstab(df.Employment, df.OrgSize.isna(), normalize="index")` before you decide.
- Whether to include the ~72 multi-hot tech columns at all.

`y` should be `np.log1p(...)`. Be sure you can say why.

In [ ]:
CAT_UNORDERED = [
    "Employment",      # why: also the key to every missing-value pattern below, so the model needs it to read those NaNs correctly
    "DevType",         # why: all 32 levels kept -- collapsing the 11 rare ones (85 rows) moved MAE by $95 and R2_log by -0.0002, so the code would buy nothing
    "RemoteWork",      # why: 5 levels with no defensible rank -- "Your choice" sits neither above nor below "Hybrid"
    "Industry",        # why: 15 levels, no natural order
    "Country",         # why: the strongest single predictor, and 130 levels is well under HGB's 255-category ceiling, so it needs no grouping
    "ICorPM",          # why: kept from clean(); its NaN doubles as a not-applicable marker that Employment already explains
    "currency_code",   # why: puts the Eurozone in one level that Country splits 20 ways -- measured MAE -$287, R2_log +0.005
]

def build_features(df):
    num = pd.DataFrame(index=df.index)                      # why: gather the numeric block in one place so the concat below is the only thing that assembles X
    num["WorkExp"] = df.WorkExp                             # why: already years on a real scale; HGB splits on NaN natively, so the 29 blanks need no imputer
    num["YearsCode"] = df.YearsCode                         # why: same, 13 blanks
    num["EdLevel_ord"] = ordinal(df.EdLevel, ED_ORDER)      # why: 41 rows -> NaN (36 "Other (please specify):" + 5 blank), which is right -- neither is a rung on the ladder
    num["Age_ord"] = ordinal(df.Age, AGE_ORDER)             # why: the 5 "Prefer not to say" rows -> NaN
    num["OrgSize_ord"] = ordinal(df.OrgSize, ORG_ORDER)     # why: 450 rows -> NaN (411 blank + 39 "I don't know"), and the crosstab shows Employment explains nearly all of them
    X = pd.concat([
        df[CAT_UNORDERED].astype("category"),               # why: NaN stays NaN -- a literal "Missing" level scored identically to 4 decimals, because HGB already gives NaN its own bin
        num,
        multi_hot(df.LanguageHaveWorkedWith, "lang_"),      # why: +42 columns, and the tech block is the biggest single win measured (MAE -$1,798, R2_log +0.041)
        multi_hot(df.DatabaseHaveWorkedWith, "db_"),        # why: +30 columns; caveat -- fillna("") leaves 882 non-answers indistinguishable from "uses no database"
    ], axis=1)
    # ResponseId and annual_salary_usd are simply never selected. That is how clean() gets to keep
    # ResponseId for section 9 without it ever reaching the model.
    y = np.log1p(df.annual_salary_usd.values)               # why: takes the cleaned target's skew from +1.36 to -0.52, so neither tail dominates the squared error, and error in log space is proportional -- which is how a salary miss is actually judged
    return X, y, [str(t) == "category" for t in X.dtypes]   # why: cat_mask is read off the dtypes instead of hand-written, so it cannot drift out of sync with X's columns

X, y, cat_mask = build_features(df)
print(f"X: {X.shape[0]:,} rows x {X.shape[1]} features")

In [ ]:
# ordinal() maps any unmatched string to NaN, so one wrong character in a 60-char level name turns
# a whole column into missing data without raising. These check both directions of that mistake.

for _name, _order, _col in [("ED_ORDER", ED_ORDER, df.EdLevel),
                            ("ORG_ORDER", ORG_ORDER, df.OrgSize),
                            ("AGE_ORDER", AGE_ORDER, df.Age)]:
    assert len(_order) == len(set(_order)), f"{_name} lists a level twice"
    assert not set(_order) - set(_col.dropna()), \
        f"{_name} contains levels the data does not (typo?): {set(_order) - set(_col.dropna())}"

# The levels left out on purpose, named explicitly. A new survey level then fails here instead of
# silently becoming NaN in a future run.
assert set(df.EdLevel.dropna()) - set(ED_ORDER) == {"Other (please specify):"}
assert set(df.OrgSize.dropna()) - set(ORG_ORDER) == {"I don’t know"}
assert set(df.Age.dropna()) - set(AGE_ORDER) == {"Prefer not to say"}

assert len(X) == len(y) == len(df), f"row count changed: X={len(X)} y={len(y)} df={len(df)}"
assert len(cat_mask) == X.shape[1], f"cat_mask is {len(cat_mask)} long but X has {X.shape[1]} columns"
assert X.columns.is_unique, "duplicate column name -- a lang_/db_ token collision"
assert not np.isnan(y).any(), "log1p produced NaN, so a non-positive salary got past the trim"
assert 80 <= X.shape[1] <= 90, f"{X.shape[1]} features -- the tech block changed size, recheck it"

print(f"X: {X.shape[0]:,} rows x {X.shape[1]} features  "
      f"({sum(cat_mask)} categorical, "
      f"{sum(c.startswith(('lang_', 'db_')) for c in X.columns)} tech, "
      f"{X.shape[1] - sum(cat_mask) - sum(c.startswith(('lang_', 'db_')) for c in X.columns)} numeric)")
print("y: log1p dollars, range "
      f"{y.min():.2f}-{y.max():.2f}  (= ${np.expm1(y.min()):,.0f}-${np.expm1(y.max()):,.0f})")
print("ordinal -> NaN:",
      {c: int(X[c].isna().sum()) for c in ["EdLevel_ord", "Age_ord", "OrgSize_ord"]})

---
# 5. Baselines

**Build these before the model.** If you fit the model first you'll anchor on its number and the
baseline becomes a formality instead of a test.

A metric with nothing to compare against is not a result.

### TODO(you): two baselines

**Baseline 0 — global median.** Predict the same number for every respondent. Ignore all features.

**Baseline 1 — country median.** Predict the median salary of the respondent's country.

Baseline 1 has a trap. If you compute each country's median using *all* rows, then every test row's
own salary helped produce the number it gets scored against — the baseline is inflated and your
model looks better than it is by comparison.

So compute it **out-of-fold**: use the same `KFold` splitter as your model, and for each fold
compute the medians using training rows only. Handle countries that appear in the test fold but not
the training fold.

In [ ]:
kf = KFold(N_FOLDS, shuffle=True, random_state=RANDOM_SEED)   # why: one splitter object, reused by section 6, so every model below is scored on identical folds

country = df.Country.values             # why: .values because KFold yields positions, and positions index arrays, not labels

oof_global = np.full(len(y), np.nan)    # why: NaN not zeros -- an unfilled slot then fails an assert instead of passing as a plausible prediction
oof_country = np.full(len(y), np.nan)
n_fallback = 0

for tr, te in kf.split(X):
    train_median = np.median(y[tr])                               # why: the median of log1p equals log1p of the median, so working in log space costs nothing here
    oof_global[te] = train_median                                 # why: baseline 0 -- ignores every feature, exists to put R2_log ~ 0 on the board
    by_country = pd.Series(y[tr]).groupby(country[tr]).median()   # why: TRAIN rows only -- computed on all 4,414 this flatters R2_log by +0.047, because each test row then helped set the bar it is judged against
    hit = pd.Series(country[te]).map(by_country)                  # why: NaN wherever this fold's training rows held nobody from that country
    n_fallback += int(hit.isna().sum())
    oof_country[te] = hit.fillna(train_median).values             # why: the global median is the only estimate left for a country the model never saw

assert not np.isnan(oof_global).any(), "a row never landed in a test fold"
assert not np.isnan(oof_country).any(), f"{np.isnan(oof_country).sum()} rows kept a NaN prediction"

print(f"baseline 0: {len(np.unique(oof_global))} distinct predictions across {N_FOLDS} folds, "
      f"${np.expm1(oof_global).min():,.0f}-${np.expm1(oof_global).max():,.0f}")
print(f"baseline 1: {n_fallback} rows ({n_fallback / len(y):.1%}) fell back to the global median")


---
# 6. Model

### TODO(you): fit and cross-validate

`HistGradientBoostingRegressor` takes categoricals and NaN natively, so you need no encoder and no
imputer. Pass `categorical_features=cat_mask`.

Use `cross_val_predict` with the **same** `KFold` object as your baseline, so the comparison is
apples to apples. Every prediction it returns comes from a model that never saw that row.

Optional, and worth it: also fit a `Ridge` on one-hot features. It will lose. Being able to say
*why* it loses — salary effects are interaction-heavy and a linear model can only add — is a
better answer than any tuning you could do.

In [ ]:
model = HistGradientBoostingRegressor(
    categorical_features=cat_mask,   # why: read off X's dtypes in section 4, so it cannot drift from the columns it describes
    random_state=RANDOM_SEED,
    max_iter=300,                    # why: the same settings the section 3 trim test used, so that experiment really did hold the model fixed
    learning_rate=0.06,
)
oof_model = cross_val_predict(model, X, y, cv=kf)   # why: the same kf object as both baselines -- identical folds, so section 7 compares like with like

assert oof_model.shape == y.shape, f"{oof_model.shape} predictions for {y.shape} targets"
print(f"out-of-fold predictions for {len(oof_model):,} rows, "
      f"${np.expm1(oof_model).min():,.0f}-${np.expm1(oof_model).max():,.0f}")


---
# 7. Evaluation

Metrics in dollars, not log-dollars. Nobody can interpret "MAE of 0.42".

In [ ]:
def report(name, y_true_log, y_pred_log):
    """Back-transform to dollars and return the four metrics."""
    actual, pred = np.expm1(y_true_log), np.expm1(y_pred_log)
    ss_res = ((y_true_log - y_pred_log) ** 2).sum()
    ss_tot = ((y_true_log - y_true_log.mean()) ** 2).sum()
    return {
        "model":  name,
        "MAE":    np.abs(actual - pred).mean(),
        "MedAE":  np.median(np.abs(actual - pred)),
        "RMSE":   np.sqrt(((actual - pred) ** 2).mean()),
        "R2_log": 1 - ss_res / ss_tot,
    }

results = pd.DataFrame([
    report("global median",  y, oof_global),    # why: the floor -- R2_log near 0 is the check that the metric itself is calibrated
    report("country median", y, oof_country),   # why: the real opponent, one feature, and it closes most of the gap
    report("HGB",            y, oof_model),     # why: 84 features, scored on the same folds as the two above
]).set_index("model")
results.round({"MAE": 0, "MedAE": 0, "RMSE": 0, "R2_log": 3})


In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.scatter(np.expm1(y), np.expm1(oof_model), s=4, alpha=.25)
lims = [5e3, 6e5]
ax.plot(lims, lims, "r--", lw=1)            # perfect prediction line
ax.set(xscale="log", yscale="log", xlim=lims, ylim=lims,
       xlabel="actual USD", ylabel="predicted USD")


### TODO(you): inspect the 10 worst predictions by hand

Ten minutes here beats an hour of hyperparameter tuning, and it produces things you can actually
say out loud.

Build a frame of `Country`, `WorkExp`, `DevType`, actual, predicted, absolute error. Sort by error.
Look at the top 10 and ask: is there a pattern, or are these just genuinely unpredictable people?

Write down what you find, including "no pattern" if that's the honest answer.

In [ ]:
worst = pd.DataFrame({
    "Country": df.Country.values,      # why: .values throughout -- y and oof_model are positional arrays, and df's index still has clean()'s gaps in it
    "WorkExp": df.WorkExp.values,
    "DevType": df.DevType.values,
    "actual": np.expm1(y),
    "predicted": np.expm1(oof_model),
})
worst["abs_error"] = (worst.actual - worst.predicted).abs()
worst.nlargest(10, "abs_error").round(0)


---
# 8. Limitations

### TODO(you): write these honestly

Three prompts. Two or three sentences each, no hedging.

1. **How much of this is just `Country`?** Run permutation importance and put the number here.
   `from sklearn.inspection import permutation_importance`
2. **How wrong is the model in practice?** Express MAE as a percentage of median salary. Is that
   good enough to make a decision with? Say so either way.
3. **Who is this model actually valid for?** Every row here is someone who took a survey *and*
   chose to disclose their pay. What does that exclude?

### TODO(you): what you'd do differently

Four things, ranked. Be specific — "more feature engineering" is not an answer, "pull `CompFreq`
from the full public survey to audit the low tail" is.

---
# 9. Save outputs

In [ ]:
# TODO(you): uncomment once the pipeline runs end to end
# import json, pathlib
# pathlib.Path("outputs").mkdir(exist_ok=True)
#
# results.to_json("outputs/metrics.json", indent=2)
# pd.DataFrame({
#     "ResponseId": df.ResponseId.values,
#     "actual":     np.expm1(y),
#     "predicted":  np.expm1(oof_model),
#     "abs_error":  np.abs(np.expm1(y) - np.expm1(oof_model)),
# }).to_csv("outputs/predictions.csv", index=False)
# print("wrote outputs/metrics.json and outputs/predictions.csv")